# Wildfire Governance Simulation — Interactive Walkthrough

This notebook demonstrates the full GOMDP governance pipeline interactively.
Run cells top-to-bottom for a complete walkthrough.

**Paper:** Governance-Constrained Agentic AI: A GOMDP Framework with Blockchain-Enforced Human Oversight

**Sections:**
1. Environment setup and fire propagation
2. Belief state and risk estimation  
3. GOMDP governance predicate
4. Blockchain smart contract verification
5. Theorem 1 empirical verification
6. Theorem 2 adversarial robustness

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

print('Imports OK')

## 1. Fire Propagation Environment

In [ ]:
from wildfire_governance.simulation.grid_environment import WildfireGridEnvironment, EnvironmentConfig

# Create a 20x20 grid environment for demonstration
env = WildfireGridEnvironment(EnvironmentConfig(grid_size=20, n_timesteps=50, n_ignition_points=2))
env.reset(seed=42)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(env.heat_map, cmap='hot', vmin=0, vmax=1)
axes[0].set_title('Initial Heat Map $H_0$')
axes[1].imshow(env.fire_mask, cmap='Reds', vmin=0, vmax=1)
axes[1].set_title('Fire Mask (Ground Truth)')
axes[2].imshow(env._fuel_map, cmap='Greens', vmin=0, vmax=1)
axes[2].set_title('Fuel Map $F$')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Grid size: {env.grid_size}x{env.grid_size}')
print(f'Initial fire cells: {int(env.fire_mask.sum())}')

In [ ]:
# Simulate 20 steps of fire propagation
heat_maps = [env.heat_map.copy()]
for _ in range(20):
    obs, done, _ = env.step([])
    heat_maps.append(env.heat_map.copy())
    if done:
        break

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
steps = [0, 5, 10, 15, 20]
for ax, s in zip(axes, steps):
    ax.imshow(heat_maps[min(s, len(heat_maps)-1)], cmap='hot', vmin=0, vmax=1)
    ax.set_title(f't={s}')
    ax.axis('off')
plt.suptitle('Fire Propagation (Stochastic CA Model)', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Belief State and Two-Stage Verification

In [ ]:
from wildfire_governance.decision.belief_state import BeliefState
from wildfire_governance.simulation.sensor_models import ThermalUAVSensor, SensorReading
from wildfire_governance.verification.fusion import CrossModalFusion
from wildfire_governance.verification.bayesian_update import BayesianConfidenceUpdate

belief = BeliefState(grid_size=20)
sensor = ThermalUAVSensor(detection_probability=0.85)
rng = np.random.default_rng(42)

# Generate sensor readings and update belief
positions = [(5,5), (10,10), (15,15), (3,8), (12,4)]
readings = [sensor.observe(env.heat_map, pos, rng) for pos in positions]
belief.update(readings)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(env.heat_map, cmap='hot', vmin=0, vmax=1)
axes[0].set_title('Ground Truth Heat Map')
axes[0].axis('off')
axes[1].imshow(belief.get_belief(), cmap='YlOrRd', vmin=0, vmax=0.3)
axes[1].set_title(f'Belief State $b_t$ (entropy={belief.entropy():.2f})')
axes[1].axis('off')
plt.tight_layout()
plt.show()

# Two-stage verification (Eqs. 5-6)
fusion = CrossModalFusion(w_h=0.65, w_w=0.35)
bayesian = BayesianConfidenceUpdate(detection_probability=0.85)

heat_idx = float(env.heat_map.max())
weather_idx = float(np.clip(env._wind_field.mean() - env._humidity_field.mean() + 0.5, 0, 1))
conf1 = fusion.compute_stage1_confidence(heat_idx, weather_idx)
conf2 = bayesian.update(conf1, verification_positive=True)

print(f'Heat anomaly index:    {heat_idx:.3f}')
print(f'Weather index:         {weather_idx:.3f}')
print(f'Stage-1 Conf^(1)_t:   {conf1:.3f}  (Eq. 5)')
print(f'Stage-2 Conf^(2)_t:   {conf2:.3f}  (Eq. 6, Bayesian update)')
print(f'Escalated to HITL:    {conf2 > 0.80}')

## 3. GOMDP Governance Predicate (Definition 1)

In [ ]:
from wildfire_governance.gomdp.definition import GovernanceInvariantMDP

gomdp = GovernanceInvariantMDP(tau=0.80)

# Case 1: High confidence + human approval → APPROVED
r1 = gomdp.step_alert_action(confidence=0.92, human_approval=True, validator_signature_valid=True)
print(f'Case 1 (conf=0.92, HA=True,  sig=valid):  blocked={r1.blocked}, state={r1.contract_state.name}')

# Case 2: High confidence but NO human approval → BLOCKED
r2 = gomdp.step_alert_action(confidence=0.92, human_approval=False, validator_signature_valid=True)
print(f'Case 2 (conf=0.92, HA=False, sig=valid):  blocked={r2.blocked}, state={r2.contract_state.name}')

# Case 3: Low confidence → BLOCKED
r3 = gomdp.step_alert_action(confidence=0.70, human_approval=True, validator_signature_valid=True)
print(f'Case 3 (conf=0.70, HA=True,  sig=valid):  blocked={r3.blocked}, state={r3.contract_state.name}')

# Case 4: Invalid signature → BLOCKED
r4 = gomdp.step_alert_action(confidence=0.95, human_approval=True, validator_signature_valid=False)
print(f'Case 4 (conf=0.95, HA=True,  sig=invalid): blocked={r4.blocked}, state={r4.contract_state.name}')

print(f'\nTheorem 1 holds: only Case 1 produces an alert.')
print(f'Blocked attempts: {gomdp.get_violation_count()} / 4')

## 4. Blockchain Smart Contract Verification

In [ ]:
from wildfire_governance.blockchain.crypto_utils import generate_key_pair, sign
from wildfire_governance.blockchain.smart_contract import GovernanceSmartContract
from wildfire_governance.blockchain.transaction import build_transaction

# Generate validator key pair
private_key, public_key = generate_key_pair()
contract = GovernanceSmartContract(tau=0.80)

# Build an anomaly transaction
tx = build_transaction(
    event_id='notebook_demo_evt_001',
    geo_boundary=(10, 10, 12, 12),
    confidence_score=0.91,
    sensor_readings={'heat': 0.91, 'weather': 0.68, 'frp': 45.2}
)
print(f'Transaction hash: {tx.transaction_hash[:24]}...')
print(f'Evidence hash:    {tx.evidence_hash[:24]}...')
print(f'Nonce:            {tx.nonce[:16]}...')

# Sign with validator private key
signature = sign(tx.to_bytes(), private_key)

# Smart contract verifies atomically
result = contract.verify_and_execute(tx, signature, public_key)
print(f'\nSmart contract result:')
print(f'  Alert enabled:  {result.alert_enabled}')
print(f'  Contract state: {result.contract_state.name}')
print(f'  Confidence OK:  {result.confidence_ok}')
print(f'  Signature OK:   {result.signature_ok}')
print(f'  Certificate:    {result.cert[:24] if result.cert else None}...')
print(f'  Audit log size: {len(contract.audit_log)} entries')

## 5. Theorem 1 Empirical Verification

In [ ]:
from wildfire_governance.gomdp.invariant_checker import GovernanceInvariantChecker
from wildfire_governance.rl.gomdp_env import GOMMDPGymEnv
from wildfire_governance.simulation.grid_environment import EnvironmentConfig

checker = GovernanceInvariantChecker(tau=0.80)
env_cfg = EnvironmentConfig(grid_size=10, n_timesteps=50)
gomdp_env = GOMMDPGymEnv(config=env_cfg, n_uavs=3, enable_governance=True)

n_episodes = 10
results = []
for ep in range(n_episodes):
    obs, _ = gomdp_env.reset(seed=ep)
    done = False
    while not done:
        # RANDOM policy — worst case for Theorem 1
        action = np.random.default_rng(ep).integers(0, 25, size=3)
        obs, _, terminated, truncated, _ = gomdp_env.step(action)
        done = terminated or truncated
    traj = gomdp_env.get_trajectory()
    report = checker.check_trajectory(traj)
    results.append(report.theorem1_satisfied)

compliance = sum(results) / len(results)
print(f'Theorem 1 Empirical Verification (Random Policy)')
print(f'Episodes:          {n_episodes}')
print(f'Compliant:         {sum(results)}/{n_episodes}')
print(f'Compliance rate:   {compliance:.1%}')
print(f'Theorem 1 holds:   {compliance == 1.0} ← must always be True')

## 6. Theorem 2: Adversarial Robustness

In [ ]:
from wildfire_governance.gomdp.breach_probability import (
    compute_breach_probability_gomdp,
    compute_breach_probability_centralized,
    generate_comparison_table,
)

# Reproduce Table V (theoretical breach probabilities)
table = generate_comparison_table(k_values=[7, 11, 15], p_c_values=[0.1, 0.2, 0.3])
print('Theorem 2: Breach Probability Comparison')
print(table.to_string(index=False))

# Plot P_breach vs p_c for GOMDP (k=7, f=2) vs centralized
p_cs = np.linspace(0.01, 0.35, 50)
p_gomdp = [compute_breach_probability_gomdp(7, 2, float(p)) for p in p_cs]
p_central_injection = [1.0] * len(p_cs)  # Direct injection: always succeeds centralized

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p_cs, p_gomdp, 'b-o', markersize=3, label='GOMDP (k=7, f=2)', linewidth=2)
ax.axhline(1.0, color='red', linestyle='--', label='Centralized (direct injection)')
ax.fill_between(p_cs, p_gomdp, 1.0, alpha=0.15, color='green', label='Safety margin')
ax.set_xlabel('Validator compromise probability $p_c$')
ax.set_ylabel('Breach probability $P_{\\mathrm{breach}}$')
ax.set_title('Theorem 2: Adversarial Robustness (GOMDP vs. Centralized)')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

print(f'\nKey result (paper Table V):')
print(f'  k=7, f=2, p_c=0.3: P_breach^GOMDP = {compute_breach_probability_gomdp(7,2,0.3):.3f}')
print(f'  Centralized (direct injection):   P_breach^central = 1.000')
print(f'  GOMDP is {1.0/compute_breach_probability_gomdp(7,2,0.3):.1f}x safer than centralized')